# Multi-source demo (v3) — SRU + Boiler_15, all three gaps closed

**Audience:** compliance reviewers. `Run All`, read the tables.

This runs the **real v3 clerk** on two realistic sources built from your uploaded
`list_a_multi.csv` (manual log) and `list_b_multi.csv` (Seeq detections), unchanged.
The roster (`roster.csv`) registers each analyzer's **source**, **obligation (pollutant)**,
and **diluent link**:

| Analyzer | Source | Obligation | Diluent |
|----------|--------|-----------|---------|
| SRU-O2   | SRU       | O2  | *is the O2 diluent* |
| SRU-SO2  | SRU       | SO2 | corrected → SRU-O2 |
| B15-O2   | Boiler_15 | O2  | *is the O2 diluent* |
| B15-NOx  | Boiler_15 | NOx | corrected → B15-O2 |
| B15-CO   | Boiler_15 | CO  | corrected → B15-O2 |

**What each labeled scenario proves**

- **M1 / M2 — diluent propagation:** an O2 outage invalidates the pollutants corrected to
  it (SRU O2 → SO2; B15 O2 → NOx **and** CO), even with clean own signals.
- **M3 — per-obligation rollup:** a B15 NOx-only outage reports the **NOx obligation** down
  while O2 and CO stay covered — a valid monitor for one pollutant never masks another.
- **M4 — T4d (source-agnostic):** an SRU O2 outage logged **manually with no Seeq capsule**
  still propagates to SO2, identical to a detected one.
- List C carries a **record for every propagated dependent**, so the standalone record
  matches the hourly grid.

*No live connections: reads the three CSVs in this folder, nothing else.*

In [1]:
# Setup — imports and paths. The clerk_v3 package must sit next to this file.
import csv, os, sys
from datetime import datetime, timedelta, timezone
from pathlib import Path
import pandas as pd

NB = Path.cwd()
sys.path.insert(0, str(NB if (NB / "clerk").exists() else NB / "clerk_v3"))
from clerk.grid import build_grid, source_down_hours
from clerk.listc import build_list_c
from clerk.schemas import (Capsule, CellValid, Event, EventType,
                           OperatingWindow, SiteConfig, read_analyzer_units)
import inspect
print("engine:", inspect.getsourcefile(build_grid))
assert "clerk" in inspect.getsourcefile(build_grid)
pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 200)
DAY = datetime(2026, 4, 1, tzinfo=timezone.utc)
def hh(d): return d.strftime("%H:%M")

engine: /home/user/SeeQ-Flare-Project/clerk_v3/clerk/grid.py


In [2]:
# Roster — registers the two sources, their obligations, and diluent links.
ROSTER = read_analyzer_units(NB / "roster.csv")
obl = {a.Analyzer: a.Obligation for a in ROSTER}
display(pd.DataFrame([{"Analyzer": a.Analyzer, "Source": a.Unit, "Obligation": a.Obligation,
                       "Role": a.DiluentsRole, "Diluent basis": a.DiluentBasis or "—"}
                      for a in ROSTER]))

,Analyzer,Source,Obligation,Role,Diluent basis
0,SRU-SO2,SRU,SO2,diluent-corrected,SRU-O2
1,SRU-O2,SRU,O2,diluent,—
2,B15-NOx,Boiler_15,NOx,diluent-corrected,B15-O2
3,B15-CO,Boiler_15,CO,diluent-corrected,B15-O2
4,B15-O2,Boiler_15,O2,diluent,—


In [3]:
# Loader — read YOUR files unchanged, grouped by the Scenario column.
def _dt(s): return datetime.fromisoformat(s.replace("Z","+00:00")) if str(s).strip() else None
EV, CA = {}, {}
adf = pd.read_csv(NB / "list_a_multi.csv").fillna("")
for _, r in adf.iterrows():
    EV.setdefault(r["Scenario"], []).append(Event(
        EventID=str(r["EventID"]), EventType=EventType(r["Type"]),
        TargetEventID=str(r["TargetEventID"]) or None,
        ExtentStartUTC=_dt(str(r["StartUTC"])), ExtentEndUTC=_dt(str(r["EndUTC"])),
        AnalyzerCEMIDs=[str(r["Analyzer"])], Category=str(r["Category"]),
        ReasonCode=str(r["ReasonCode"]), Actor=str(r["Actor"]),
        ActedAt=_dt(str(r["ActedAtUTC"])), Reason=str(r["Note"]),
        CorrectiveAction=str(r["CorrectiveAction"]), DetectionClass=str(r["DetectionClass"])))
bdf = pd.read_csv(NB / "list_b_multi.csv").fillna("")
for _, r in bdf.iterrows():
    CA.setdefault(r["Scenario"], []).append(Capsule(
        str(r["Analyzer"]), str(r["DetectionClass"]),
        _dt(str(r["CapsuleStartUTC"])), _dt(str(r["CapsuleEndUTC"]))))
print(f"list_a_multi.csv: {len(adf)} rows;  list_b_multi.csv: {len(bdf)} rows")
print("scenarios:", ", ".join(sorted(set(adf.Scenario) | set(bdf.Scenario))))

list_a_multi.csv: 7 rows;  list_b_multi.csv: 3 rows
scenarios: M1, M2, M3, M4, M5, M6


In [4]:
# Per-scenario runner + display helpers.
CFG = SiteConfig(SiteTimeZoneIANA="America/New_York", LookbackMonths=8,
                 LateXThresholdDays=7, JitterToleranceMin=5,
                 PartialOperatingHourApplicability={},
                 ReasonParagraphMap={"MM-01": "(i)", "QA-01": "(iii)"})
START, END = DAY + timedelta(hours=5), DAY + timedelta(hours=16)
OPERATING = [OperatingWindow(u, START, END) for u in {a.Unit for a in ROSTER}]
RESULTS, LISTC_ROWS = [], []

def run(scenario):
    ev, ca = EV.get(scenario, []), CA.get(scenario, [])
    cells = build_grid(ev, ca, OPERATING, ROSTER, [], CFG, START, END)
    recs = build_list_c(ev, ca, cells, 5, analyzer_units=ROSTER)
    for r in recs:
        LISTC_ROWS.append({"Scenario": scenario, "Analyzer": r.Analyzer, "Obligation": obl[r.Analyzer],
            "State": r.State, "Resolved": " + ".join(f"{s:%H:%M}-{e:%H:%M}" for s,e in r.ResolvedWindows),
            "Source used": r.SourceUsed, "Reason": r.ReasonCode, "Note": r.Note,
            "Approver": r.ApproverName, "Down hours": ";".join(hh(h) for h in r.DownHours),
            "Provenance": ";".join(r.ContributingRecords)})
    return ev, ca, cells, recs

def show_inputs(ev, ca):
    a = pd.DataFrame([{"Event": e.EventID, "Analyzer": e.AnalyzerCEMIDs[0],
        "From": hh(e.ExtentStartUTC), "To": hh(e.ExtentEndUTC), "Reason": e.ReasonCode,
        "Note": e.Reason} for e in ev]) if ev else pd.DataFrame([{"(no List A rows)": ""}])
    b = pd.DataFrame([{"Analyzer": c.Analyzer, "Class": c.DetectionClass,
        "From": hh(c.CapsuleStartUTC), "To": hh(c.CapsuleEndUTC)} for c in ca])         if ca else pd.DataFrame([{"(no List B capsules)": ""}])
    print("List A (manual log):"); display(a); print("List B (Seeq detections):"); display(b)

def down_map(cells, unit):
    out = {}
    for c in cells:
        if c.Analyzer.startswith(unit.split("_")[0]) and c.Valid is CellValid.invalid:
            out.setdefault(c.Analyzer, []).append(hh(c.HourStartUTC))
    return out

def show_records(recs):
    rows = [{"Analyzer": r.Analyzer, "Obl": obl[r.Analyzer], "State": r.State,
             "Resolved": " + ".join(f"{s:%H:%M}-{e:%H:%M}" for s,e in r.ResolvedWindows),
             "Source used": r.SourceUsed, "Down": ";".join(hh(h) for h in r.DownHours),
             "Note": r.Note} for r in recs if r.DownHours]
    display(pd.DataFrame(rows) if rows else pd.DataFrame([{"(no downtime records)": ""}]))

def show_rollup(cells, unit):
    roll = source_down_hours(ROSTER, cells)
    rows = [{"Source": u, "Obligation": o, "Obligation down hours": ";".join(hh(h) for h in hrs)}
            for (u, o), hrs in sorted(roll.items()) if u == unit]
    display(pd.DataFrame(rows) if rows else pd.DataFrame([{f"{unit}: no obligation down": ""}]))

def check(name, actual, expected):
    ok = actual == expected
    RESULTS.append({"Check": name, "Expected": str(expected), "Actual": str(actual),
                    "Result": "PASS" if ok else "*** FAIL ***"})
    print(("PASS " if ok else ">>> FAIL  ") + name)
    if not ok: print(f"      expected {expected}\n      actual   {actual}")

---
## M1 — SRU O2 outage → propagates to SO2 (detected + logged)
Seeq **and** the tech both report SRU-O2 down 09:00–11:00. SRU-SO2 is corrected to O2, so
it is invalid those hours even though its own signal is clean.

In [5]:
ev, ca, cells, recs = run("M1")
show_inputs(ev, ca)
print("Per-analyzer down hours (SRU):", down_map(cells, "SRU"))
print("Obligation rollup (SRU):"); show_rollup(cells, "SRU")
print("List C:"); show_records(recs)
check("M1 SRU-O2 down", down_map(cells, "SRU").get("SRU-O2"), ["09:00","10:00"])
check("M1 SRU-SO2 propagated down", down_map(cells, "SRU").get("SRU-SO2"), ["09:00","10:00"])
check("M1 SO2 has a propagated List C record",
      any(r.Analyzer=="SRU-SO2" and "propagated" in r.SourceUsed for r in recs), True)

List A (manual log):


,Event,Analyzer,From,To,Reason,Note
0,MA1,SRU-O2,09:00,11:00,MM-01,O2 analyzer drift alarm after vent line blockage


List B (Seeq detections):


,Analyzer,Class,From,To
0,SRU-O2,status-offline,09:00,11:00


Per-analyzer down hours (SRU): {'SRU-SO2': ['09:00', '10:00'], 'SRU-O2': ['09:00', '10:00']}
Obligation rollup (SRU):


,Source,Obligation,Obligation down hours
0,SRU,O2,09:00;10:00
1,SRU,SO2,09:00;10:00


List C:


,Analyzer,Obl,State,Resolved,Source used,Down,Note
0,SRU-O2,O2,auto-approved,09:00-11:00,concurrence (A=B),09:00;10:00,O2 analyzer drift alarm after vent line blockage
1,SRU-SO2,SO2,auto-approved,09:00-11:00,diluent-propagated (from SRU-O2),09:00;10:00,Invalid because diluent SRU-O2 was down: O2 analyzer drift alarm after vent line blockage


PASS M1 SRU-O2 down
PASS M1 SRU-SO2 propagated down
PASS M1 SO2 has a propagated List C record


---
## M2 — Boiler_15 O2 outage → propagates to BOTH NOx and CO
One diluent outage, two dependents.

In [6]:
ev, ca, cells, recs = run("M2")
show_inputs(ev, ca)
print("Per-analyzer down hours (Boiler_15):", down_map(cells, "B15"))
print("Obligation rollup (Boiler_15):"); show_rollup(cells, "Boiler_15")
print("List C:"); show_records(recs)
check("M2 B15-O2 down", down_map(cells, "B15").get("B15-O2"), ["09:00","10:00"])
check("M2 B15-NOx propagated", down_map(cells, "B15").get("B15-NOx"), ["09:00","10:00"])
check("M2 B15-CO propagated", down_map(cells, "B15").get("B15-CO"), ["09:00","10:00"])

List A (manual log):


,Event,Analyzer,From,To,Reason,Note
0,MA2,B15-O2,09:00,11:00,MM-01,Boiler 15 O2 chiller failed


List B (Seeq detections):


,Analyzer,Class,From,To
0,B15-O2,status-offline,09:00,11:00


Per-analyzer down hours (Boiler_15): {'B15-NOx': ['09:00', '10:00'], 'B15-CO': ['09:00', '10:00'], 'B15-O2': ['09:00', '10:00']}
Obligation rollup (Boiler_15):


,Source,Obligation,Obligation down hours
0,Boiler_15,CO,09:00;10:00
1,Boiler_15,NOx,09:00;10:00
2,Boiler_15,O2,09:00;10:00


List C:


,Analyzer,Obl,State,Resolved,Source used,Down,Note
0,B15-CO,CO,auto-approved,09:00-11:00,diluent-propagated (from B15-O2),09:00;10:00,Invalid because diluent B15-O2 was down: Boiler 15 O2 chiller failed
1,B15-NOx,NOx,auto-approved,09:00-11:00,diluent-propagated (from B15-O2),09:00;10:00,Invalid because diluent B15-O2 was down: Boiler 15 O2 chiller failed
2,B15-O2,O2,auto-approved,09:00-11:00,concurrence (A=B),09:00;10:00,Boiler 15 O2 chiller failed


PASS M2 B15-O2 down
PASS M2 B15-NOx propagated
PASS M2 B15-CO propagated


---
## M3 — per-obligation rollup: NOx-only outage is NOT masked
B15-NOx is down 06:00–08:00; O2 and CO are valid. The **NOx obligation** must report down
at 06:00/07:00 — the pre-v3 per-unit intersection wrongly reported nothing (valid CO/O2
masked it). This is the Gap #1 discriminator.

In [7]:
ev, ca, cells, recs = run("M3")
show_inputs(ev, ca)
print("Per-analyzer down hours (Boiler_15):", down_map(cells, "B15"))
print("Obligation rollup (Boiler_15):"); show_rollup(cells, "Boiler_15")
print("List C:"); show_records(recs)
roll = source_down_hours(ROSTER, cells)
check("M3 NOx obligation down 06/07",
      [hh(h) for h in roll.get(("Boiler_15","NOx"), [])], ["06:00","07:00"])
check("M3 O2 obligation NOT down (not masked away, and not falsely down)",
      ("Boiler_15","O2") in roll, False)
check("M3 CO obligation NOT down", ("Boiler_15","CO") in roll, False)

List A (manual log):


,Event,Analyzer,From,To,Reason,Note
0,MA3,B15-NOx,06:00,08:00,MM-01,NOx analyzer plugged sample filter


List B (Seeq detections):


,Analyzer,Class,From,To
0,B15-NOx,status-offline,06:00,08:00


Per-analyzer down hours (Boiler_15): {'B15-NOx': ['06:00', '07:00']}
Obligation rollup (Boiler_15):


,Source,Obligation,Obligation down hours
0,Boiler_15,NOx,06:00;07:00


List C:


,Analyzer,Obl,State,Resolved,Source used,Down,Note
0,B15-NOx,NOx,auto-approved,06:00-08:00,concurrence (A=B),06:00;07:00,NOx analyzer plugged sample filter


PASS M3 NOx obligation down 06/07
PASS M3 O2 obligation NOT down (not masked away, and not falsely down)
PASS M3 CO obligation NOT down


---
## M4 — T4d: manual-only SRU O2 outage (no Seeq capsule) still propagates
The SRU-O2 outage 13:00–15:00 is logged by the tech with **no detection** (List B has no
capsule for it). Propagation must still fire to SO2 — source-agnostic (D7).

In [8]:
ev, ca, cells, recs = run("M4")
show_inputs(ev, ca)
print("List B for M4 (should be empty — manual only):", CA.get("M4", []))
print("Per-analyzer down hours (SRU):", down_map(cells, "SRU"))
print("List C:"); show_records(recs)
check("M4 SRU-O2 down (manual only)", down_map(cells, "SRU").get("SRU-O2"), ["13:00","14:00"])
check("M4 SRU-SO2 propagated from a MANUAL outage (T4d)",
      down_map(cells, "SRU").get("SRU-SO2"), ["13:00","14:00"])
check("M4 SO2 propagated record cites the manual O2 outage",
      any(r.Analyzer=="SRU-SO2" and "propagated" in r.SourceUsed for r in recs), True)

List A (manual log):


,Event,Analyzer,From,To,Reason,Note
0,MA4,SRU-O2,13:00,15:00,MM-01,O2 down - maintenance drift on O2. Logged by tech; no auto detection


List B (Seeq detections):


,(no List B capsules)
0,


List B for M4 (should be empty — manual only): []
Per-analyzer down hours (SRU): {'SRU-SO2': ['13:00', '14:00'], 'SRU-O2': ['13:00', '14:00']}
List C:


,Analyzer,Obl,State,Resolved,Source used,Down,Note
0,SRU-O2,O2,auto-approved,13:00-15:00,A only,13:00;14:00,O2 down - maintenance drift on O2. Logged by tech; no auto detection
1,SRU-SO2,SO2,auto-approved,13:00-15:00,diluent-propagated (from SRU-O2),13:00;14:00,Invalid because diluent SRU-O2 was down: O2 down - maintenance drift on O2. Logged by ...


PASS M4 SRU-O2 down (manual only)
PASS M4 SRU-SO2 propagated from a MANUAL outage (T4d)
PASS M4 SO2 propagated record cites the manual O2 outage


---
## M5 / M6 — additional realistic entries (QA calibrations, touch-up)
M5: quarterly CGA on B15 NOx and CO (a maintenance hour). M6: a short SO2 touch-up cal that
leaves enough valid time in the hour. Shown for completeness — not gap demonstrations.

In [9]:
for sc in ("M5", "M6"):
    ev, ca, cells, recs = run(sc)
    print(f"=== {sc} ===")
    show_inputs(ev, ca)
    print("Down hours:", {**down_map(cells,'SRU'), **down_map(cells,'B15')})
    show_records(recs)

=== M5 ===
List A (manual log):


,Event,Analyzer,From,To,Reason,Note
0,MA5,B15-NOx,09:00,10:00,QA-01,Quarterly CGA on Boiler 15 NOx
1,MA6,B15-CO,09:00,10:00,QA-01,Quarterly CGA on Boiler 15 CO


List B (Seeq detections):


,(no List B capsules)
0,


Down hours: {'B15-NOx': ['09:00'], 'B15-CO': ['09:00']}


...

,Event,Analyzer,From,To,Reason,Note
0,MA7,SRU-SO2,12:50,13:10,,Performed touch up calibration on SO2


List B (Seeq detections):


,(no List B capsules)
0,


Down hours: {}


,(no downtime records)
0,


---
## Scoreboard and the exported List C

In [10]:
sb = pd.DataFrame(RESULTS)
display(sb)
fails = sb[sb.Result != "PASS"]
print(f"\n{(sb.Result=='PASS').sum()}/{len(sb)} checks passed."
      + ("" if fails.empty else f"  {len(fails)} FAILED — see above."))

,Check,Expected,Actual,Result
0,M1 SRU-O2 down,"['09:00', '10:00']","['09:00', '10:00']",PASS
1,M1 SRU-SO2 propagated down,"['09:00', '10:00']","['09:00', '10:00']",PASS
2,M1 SO2 has a propagated List C record,True,True,PASS
3,M2 B15-O2 down,"['09:00', '10:00']","['09:00', '10:00']",PASS
4,M2 B15-NOx propagated,"['09:00', '10:00']","['09:00', '10:00']",PASS
5,M2 B15-CO propagated,"['09:00', '10:00']","['09:00', '10:00']",PASS
6,M3 NOx obligation down 06/07,"['06:00', '07:00']","['06:00', '07:00']",PASS
7,"M3 O2 obligation NOT down (not masked away, and not falsely down)",False,False,PASS
8,M3 CO obligation NOT down,False,False,PASS
9,M4 SRU-O2 down (manual only),"['13:00', '14:00']","['13:00', '14:00']",PASS



12/12 checks passed.


In [11]:
# Export the enriched List C across all scenarios, next to this notebook.
out = pd.DataFrame(LISTC_ROWS)
out.to_csv(NB / "list_c_multi.csv", index=False)
from IPython.display import FileLink, display as disp
print("Output folder:", os.getcwd())
print("Wrote list_c_multi.csv  (", len(out), "records )")
print("If the file browser doesn't show it, click its REFRESH arrow, or use the link below.")
disp(FileLink(str(NB / "list_c_multi.csv"), result_html_prefix="Download List C: "))

Output folder: /home/user/SeeQ-Flare-Project/clerk_v3
Wrote list_c_multi.csv  ( 11 records )
If the file browser doesn't show it, click its REFRESH arrow, or use the link below.


/home/user/SeeQ-Flare-Project/clerk_v3/list_c_multi.csv

### The enriched List C, inline

In [12]:
out

,Scenario,Analyzer,Obligation,State,Resolved,Source used,Reason,Note,Approver,Down hours,Provenance
0,M1,SRU-O2,O2,auto-approved,09:00-11:00,concurrence (A=B),MM-01,O2 analyzer drift alarm after vent line blockage,auto,09:00;10:00,CAP:SRU-O2:status-offline:2026-04-01T09:00:00+00:00:2026-04-01T11:00:00+00:00;MA1
1,M1,SRU-SO2,SO2,auto-approved,09:00-11:00,diluent-propagated (from SRU-O2),MM-01,Invalid because diluent SRU-O2 was down: O2 analyzer drift alarm after vent line blockage,auto,09:00;10:00,DILUENT:SRU-O2;CAP:SRU-O2:status-offline:2026-04-01T09:00:00+00:00:2026-04-01T11:00:00...
2,M2,B15-CO,CO,auto-approved,09:00-11:00,diluent-propagated (from B15-O2),MM-01,Invalid because diluent B15-O2 was down: Boiler 15 O2 chiller failed,auto,09:00;10:00,DILUENT:B15-O2;CAP:B15-O2:status-offline:2026-04-01T09:00:00+00:00:2026-04-01T11:00:00...
3,M2,B15-NOx,NOx,auto-approved,09:00-11:00,diluent-propagated (from B15-O2),MM-01,Invalid because diluent B15-O2 was down: Boiler 15 O2 chiller failed,auto,09:00;10:00,DILUENT:B15-O2;CAP:B15-O2:status-offline:2026-04-01T09:00:00+00:00:2026-04-01T11:00:00...
4,M2,B15-O2,O2,auto-approved,09:00-11:00,concurrence (A=B),MM-01,Boiler 15 O2 chiller failed,auto,09:00;10:00,CAP:B15-O2:status-offline:2026-04-01T09:00:00+00:00:2026-04-01T11:00:00+00:00;MA2
5,M3,B15-NOx,NOx,auto-approved,06:00-08:00,concurrence (A=B),MM-01,NOx analyzer plugged sample filter,auto,06:00;07:00,CAP:B15-NOx:status-offline:2026-04-01T06:00:00+00:00:2026-04-01T08:00:00+00:00;MA3
6,M4,SRU-O2,O2,auto-approved,13:00-15:00,A only,MM-01,O2 down - maintenance drift on O2. Logged by tech; no auto detection,auto,13:00;14:00,MA4
7,M4,SRU-SO2,SO2,auto-approved,13:00-15:00,diluent-propagated (from SRU-O2),MM-01,Invalid because diluent SRU-O2 was down: O2 down - maintenance drift on O2. Logged by ...,auto,13:00;14:00,DILUENT:SRU-O2;MA4
8,M5,B15-CO,CO,auto-approved,09:00-10:00,A only,QA-01,Quarterly CGA on Boiler 15 CO,auto,09:00,MA6
9,M5,B15-NOx,NOx,auto-approved,09:00-10:00,A only,QA-01,Quarterly CGA on Boiler 15 NOx,auto,09:00,MA5
